# Whisper Capability

> Capability implementation for OpenAI Whisper transcription

In [ ]:
#| default_exp capability

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import os
import sys
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, Any, Optional, List, Union, ClassVar
import warnings
import shutil

import numpy as np
import torch
from fastcore.basics import patch

try:
    import whisper
    from whisper import load_model
    from whisper import transcribe
    _WHISPER_IMPORT_OK = True
except ImportError:
    _WHISPER_IMPORT_OK = False

# FFmpeg availability (replaces the retired cjm-ffmpeg-utils FFMPEG_AVAILABLE, which was
# just `shutil.which("ffmpeg") is not None`). Whisper shells out to ffmpeg to decode audio.
WHISPER_AVAILABLE = _WHISPER_IMPORT_OK and shutil.which("ffmpeg") is not None

# Stage 8 (Option C / PILLAR 1c): the tool re-bases onto ToolCapability (pure
# compute). The cache/persist bookends + the TranscriptionResult data noun moved
# OUT — the generic adapter (cjm-transcription-adapter-interface) owns the cache,
# and the result DTO lives in cjm-capability-primitives. No get_plugin_metadata.
from cjm_substrate.core.capability import ToolCapability, RELOAD_TRIGGER, EnvVarSpec
from cjm_capability_primitives.transcription import TranscriptionResult
from cjm_substrate.core.errors import (
    CapabilityInputError, CapabilityFatalError,
)
from cjm_substrate.utils.validation import (
    dict_to_config, config_to_dict, validate_config, dataclass_to_jsonschema,
    SCHEMA_TITLE, SCHEMA_DESC, SCHEMA_MIN, SCHEMA_MAX, SCHEMA_ENUM
)

# Shared torch helpers (cjm-substrate-torch-utils): release + CUDA-OOM typing + device resolution.
from cjm_substrate_torch_utils.memory import release_model
from cjm_substrate_torch_utils.oom import cuda_oom_to_capability_resource_error
from cjm_substrate_torch_utils.device import resolve_torch_device

In [ ]:
#| export
@dataclass
class WhisperCapabilityConfig:
    """Configuration for Whisper transcription capability."""
    model:str = field(
        default="base",
        metadata={
            SCHEMA_TITLE: "Model",
            SCHEMA_DESC: "Whisper model size. Larger models are more accurate but slower.",
            SCHEMA_ENUM: ["tiny", "tiny.en", "base", "base.en", "small", "small.en", 
                        "medium", "medium.en", "large", "large-v1", "large-v2", "large-v3"],
            RELOAD_TRIGGER: "model",  # CR-4: model swap requires reload
        }
    )
    device:str = field(
        default="auto",
        metadata={
            SCHEMA_TITLE: "Device",
            SCHEMA_DESC: "Device for inference (auto will use CUDA if available)",
            SCHEMA_ENUM: ["auto", "cpu", "cuda"],
            RELOAD_TRIGGER: "model",  # CR-4: device change requires model reload
        }
    )
    language:Optional[str] = field(
        default=None,
        metadata={
            SCHEMA_TITLE: "Language",
            SCHEMA_DESC: "Language code (e.g., 'en', 'es', 'fr') or None for auto-detection"
        }
    )
    task:str = field(
        default="transcribe",
        metadata={
            SCHEMA_TITLE: "Task",
            SCHEMA_DESC: "Task to perform (transcribe or translate to English)",
            SCHEMA_ENUM: ["transcribe", "translate"]
        }
    )
    temperature:float = field(
        default=0.0,
        metadata={
            SCHEMA_TITLE: "Temperature",
            SCHEMA_DESC: "Sampling temperature (0 for deterministic)",
            SCHEMA_MIN: 0.0,
            SCHEMA_MAX: 1.0
        }
    )
    temperature_increment_on_fallback:Optional[float] = field(
        default=0.2,
        metadata={
            SCHEMA_TITLE: "Temperature Increment",
            SCHEMA_DESC: "Temperature increment when falling back",
            SCHEMA_MIN: 0.0,
            SCHEMA_MAX: 1.0
        }
    )
    beam_size:int = field(
        default=5,
        metadata={
            SCHEMA_TITLE: "Beam Size",
            SCHEMA_DESC: "Beam search width",
            SCHEMA_MIN: 1,
            SCHEMA_MAX: 10
        }
    )
    best_of:int = field(
        default=5,
        metadata={
            SCHEMA_TITLE: "Best Of",
            SCHEMA_DESC: "Number of candidates when sampling",
            SCHEMA_MIN: 1,
            SCHEMA_MAX: 10
        }
    )
    patience:float = field(
        default=1.0,
        metadata={
            SCHEMA_TITLE: "Patience",
            SCHEMA_DESC: "Beam search patience factor",
            SCHEMA_MIN: 0.0,
            SCHEMA_MAX: 2.0
        }
    )
    length_penalty:Optional[float] = field(
        default=None,
        metadata={
            SCHEMA_TITLE: "Length Penalty",
            SCHEMA_DESC: "Exponential length penalty"
        }
    )
    suppress_tokens:str = field(
        default="-1",
        metadata={
            SCHEMA_TITLE: "Suppress Tokens",
            SCHEMA_DESC: "Tokens to suppress ('-1' for default)"
        }
    )
    initial_prompt:Optional[str] = field(
        default=None,
        metadata={
            SCHEMA_TITLE: "Initial Prompt",
            SCHEMA_DESC: "Optional initial prompt"
        }
    )
    condition_on_previous_text:bool = field(
        default=False,
        metadata={
            SCHEMA_TITLE: "Condition on Previous",
            SCHEMA_DESC: "Condition on previous text"
        }
    )
    fp16:bool = field(
        default=True,
        metadata={
            SCHEMA_TITLE: "FP16",
            SCHEMA_DESC: "Use FP16 (half precision) for faster inference"
        }
    )
    compression_ratio_threshold:float = field(
        default=2.4,
        metadata={
            SCHEMA_TITLE: "Compression Ratio Threshold",
            SCHEMA_DESC: "Threshold for repetition detection",
            SCHEMA_MIN: 1.0,
            SCHEMA_MAX: 10.0
        }
    )
    logprob_threshold:float = field(
        default=-1.0,
        metadata={
            SCHEMA_TITLE: "Logprob Threshold",
            SCHEMA_DESC: "Average log probability threshold"
        }
    )
    no_speech_threshold:float = field(
        default=0.6,
        metadata={
            SCHEMA_TITLE: "No Speech Threshold",
            SCHEMA_DESC: "Threshold for silence detection",
            SCHEMA_MIN: 0.0,
            SCHEMA_MAX: 1.0
        }
    )
    word_timestamps:bool = field(
        default=False,
        metadata={
            SCHEMA_TITLE: "Word Timestamps",
            SCHEMA_DESC: "Extract word-level timestamps"
        }
    )
    prepend_punctuations:str = field(
        default="\"'“¿([{-",
        metadata={
            SCHEMA_TITLE: "Prepend Punctuations",
            SCHEMA_DESC: "Punctuations to merge with next word"
        }
    )
    append_punctuations:str = field(
        default="\"'.。,，!！?？:：”)]}、",
        metadata={
            SCHEMA_TITLE: "Append Punctuations",
            SCHEMA_DESC: "Punctuations to merge with previous word"
        }
    )
    threads:int = field(
        default=0,
        metadata={
            SCHEMA_TITLE: "Threads",
            SCHEMA_DESC: "Number of threads (0 for default)",
            SCHEMA_MIN: 0
        }
    )
    model_dir:Optional[str] = field(
        default=None,
        metadata={
            SCHEMA_TITLE: "Model Directory",
            RELOAD_TRIGGER: "model",  # CR-4: changing model directory may require redownload + reload
            SCHEMA_DESC: "Directory to save/load models"
        }
    )
    compile_model:bool = field(
        default=False,
        metadata={
            SCHEMA_TITLE: "Compile Model",
            RELOAD_TRIGGER: "model",  # CR-4: torch.compile is one-shot; reload to apply
            SCHEMA_DESC: "Use torch.compile for potential speedup (requires PyTorch 2.0+)"
        }
    )

In [ ]:
#| export
class WhisperLocalCapability(ToolCapability):
    """OpenAI Whisper transcription capability (stage 8: pure-compute tool capability).

    Native-surface model (PILLAR 1c): this tool is PURE COMPUTE — `transcribe`
    loads the model, runs inference, and builds the typed `TranscriptionResult`.
    The cache-check + persistence bookends + the per-call `force` control live in
    the generic transcription adapter (cjm-transcription-adapter-interface); the
    result DTO lives in cjm-capability-primitives; identity is derived from the
    installed distribution. No `get_plugin_metadata`, no `self.storage`."""

    # CR-4: declarative reload-triggers — substrate's reconfigure_with_triggers
    # walks this config_class's dataclass fields for RELOAD_TRIGGER metadata and
    # fires the corresponding `_release_<trigger>` method on field changes.
    config_class = WhisperCapabilityConfig

    # Track 19 (CR-12 worker-env model): worker spawn env declared on the class.
    # CUDA_VISIBLE_DEVICES + OMP_NUM_THREADS are static; XDG_CACHE_HOME is templated to
    # the substrate models dir (Whisper stores downloaded weights under
    # <XDG_CACHE_HOME>/whisper). The substrate resolves + injects at Popen.
    WORKER_ENV: ClassVar[List[EnvVarSpec]] = [
        EnvVarSpec(
            name="CUDA_VISIBLE_DEVICES",
            default="0",
            label="GPU Device",
            description="Which GPU index the worker uses.",
        ),
        EnvVarSpec(
            name="OMP_NUM_THREADS",
            default="4",
            label="OpenMP Threads",
            description="Thread cap for CPU-side ops.",
        ),
        EnvVarSpec(
            name="XDG_CACHE_HOME",
            default="${CJM_MODELS_DIR}",
            label="Cache Home",
            description="XDG cache root; Whisper stores downloaded models under <XDG_CACHE_HOME>/whisper (templated to the substrate models dir).",
        ),
    ]

    def __init__(self):
        """Initialize the Whisper capability with default configuration."""
        self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
        self.config: WhisperCapabilityConfig = None
        self.model = None
        self.device = None
        self.model_dir = None

    @property
    def name(self) -> str: # Capability name identifier
        """Capability identity, derived from the installed distribution (PILLAR 1c).

        Runtime-derived: in the worker / in-env introspection `__package__`
        resolves; the manifest records the same value independently (the
        dual-mode generator reads it from the distribution)."""
        from importlib.metadata import metadata, packages_distributions
        dist = (packages_distributions().get(__package__) or [__package__.replace("_", "-")])[0]
        return metadata(dist)["Name"]

    @property
    def version(self) -> str: # Capability version string
        """Get the capability version string."""
        from cjm_capability_whisper import __version__
        return __version__

    def get_current_config(self) -> Dict[str, Any]: # Current configuration as dictionary
        """Return current configuration state."""
        if not self.config:
            return {}
        return config_to_dict(self.config)

    def get_config_schema(self) -> Dict[str, Any]: # JSON Schema for configuration
        """Return JSON Schema for UI generation."""
        return dataclass_to_jsonschema(WhisperCapabilityConfig)

    @staticmethod
    def get_config_dataclass() -> WhisperCapabilityConfig: # Configuration dataclass
        """Return dataclass describing the capability's configuration options."""
        return WhisperCapabilityConfig

    def initialize(
        self,
        config: Optional[Any] = None # Configuration dataclass, dict, or None
    ) -> None:
        """First-time setup. CR-4: the manual model/device diff-and-reload is replaced
        by declarative RELOAD_TRIGGER metadata; the substrate's reconfigure path fires
        _release_model then re-applies config via _apply_config."""
        self._apply_config(config)
        self.logger.info(f"Initialized Whisper capability with model '{self.config.model}' on device '{self.device}'")

    def transcribe(
        self,
        audio: Union[str, Path], # Path to MODEL-READY audio (converted upstream)
        **kwargs # Provenance (source_start_time/source_end_time) stamped into metadata
    ) -> TranscriptionResult: # Typed transcription output
        """Transcribe model-ready audio using Whisper — PURE COMPUTE.

        Stage 8 / PILLAR 1c: the cache-check + persistence bookends moved to the
        generic transcription adapter; this method loads the model, runs
        inference, and builds the typed result. Model params come from
        `self.config` (the CR-15 per-call override path is gone — the tool runs
        its effective config, no metadata lie); `source_start_time` /
        `source_end_time` ride the provenance kwarg channel into metadata."""
        # Validate + resolve the input path, then load the model.
        audio_path = self._prepare_audio(audio)
        self._load_model()

        # Effective config (no per-call override path).
        c = self.config
        model_name = c.model
        task = c.task
        language = c.language
        beam_size = c.beam_size
        best_of = c.best_of
        patience = c.patience
        length_penalty = c.length_penalty
        suppress_tokens = c.suppress_tokens
        initial_prompt = c.initial_prompt
        condition_on_previous_text = c.condition_on_previous_text
        fp16 = c.fp16
        compression_ratio_threshold = c.compression_ratio_threshold
        logprob_threshold = c.logprob_threshold
        no_speech_threshold = c.no_speech_threshold
        word_timestamps = c.word_timestamps
        prepend_punctuations = c.prepend_punctuations
        append_punctuations = c.append_punctuations
        temperature = c.temperature
        temp_increment = c.temperature_increment_on_fallback
        threads = c.threads

        # Prepare Whisper arguments
        whisper_args = {
            "verbose": False,
            "task": task,
            "language": language,
            "beam_size": beam_size,
            "best_of": best_of,
            "patience": patience,
            "length_penalty": length_penalty,
            "suppress_tokens": suppress_tokens,
            "initial_prompt": initial_prompt,
            "condition_on_previous_text": condition_on_previous_text,
            "fp16": fp16 and self.device == "cuda",
            "compression_ratio_threshold": compression_ratio_threshold,
            "logprob_threshold": logprob_threshold,
            "no_speech_threshold": no_speech_threshold,
            "word_timestamps": word_timestamps,
            "prepend_punctuations": prepend_punctuations,
            "append_punctuations": append_punctuations,
        }

        # Handle temperature settings
        if temp_increment is not None and temp_increment > 0:
            temperature = tuple(np.arange(temperature, 1.0 + 1e-6, temp_increment))
        else:
            temperature = [temperature]

        # Set number of threads if specified
        if threads > 0:
            torch.set_num_threads(threads)

        # Perform transcription
        self.logger.info(f"Transcribing audio with Whisper {model_name}")

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")  # Suppress Whisper warnings
                result = transcribe(
                    self.model,
                    audio_path,
                    temperature=temperature,
                    **whisper_args
                )
        except torch.cuda.OutOfMemoryError as e:
            # SG-47 Track B: CUDA OOM during Whisper inference -> typed CapabilityResourceError
            # (cjm-substrate-torch-utils); substrate's CR-7 reactive-retry reloads + retries.
            raise cuda_oom_to_capability_resource_error(
                e, label=f"during Whisper inference (model={model_name!r})",
            ) from e

        # Process segments
        segments = []
        for segment in result.get("segments", []):
            segment_data = {
                "start": segment["start"],
                "end": segment["end"],
                "text": segment["text"].strip()
            }

            # Add word timestamps if available
            if "words" in segment and word_timestamps:
                segment_data["words"] = [
                    {
                        "word": word["word"],
                        "start": word["start"],
                        "end": word["end"],
                        "probability": word.get("probability")
                    }
                    for word in segment["words"]
                ]

            segments.append(segment_data)

        # Capture provenance metadata passed via kwargs
        provenance_meta = {
            k: v for k, v in kwargs.items()
            if k in ['source_start_time', 'source_end_time']
        }

        # Create transcription result
        transcription_result = TranscriptionResult(
            text=result["text"].strip(),
            confidence=None,  # Whisper doesn't provide overall confidence
            segments=segments if segments else None,
            metadata={
                "model": model_name,
                **provenance_meta,
                "language": result.get("language", language),
                "task": task,
                "device": self.device,
                "duration": result.get("duration"),
            }
        )

        self.logger.info(f"Transcription completed: {len(result['text'].split())} words")
        return transcription_result

In [ ]:
#| export
@patch
def _apply_config(
    self:WhisperLocalCapability,
    config: Optional[Any] = None # Configuration dataclass, dict, or None
) -> None:
    """CR-4: apply config values + derive config-dependent state (device,
    model_dir). No heavy-resource work. Called by initialize (first-time) and by
    the substrate's reconfigure delta path. Model release on a model/device/
    model_dir/compile_model change is handled declaratively via RELOAD_TRIGGER
    -> _release_model (fired by the substrate BEFORE this re-applies config)."""
    self.config = dict_to_config(WhisperCapabilityConfig, config or {})

    # Resolve device ("auto" -> cuda if available, else cpu)
    self.device = resolve_torch_device(self.config.device)

    # Resolve model directory
    self.model_dir = self.config.model_dir

In [ ]:
#| export
@patch
def _release_model(self:WhisperLocalCapability) -> None:
    """CR-4: release the loaded model + free CUDA cache. RELOAD_TRIGGER target for
    model/device/model_dir/compile_model; on_disable / cleanup delegate here.
    Idempotent via cjm-substrate-torch-utils' release_model (no-op when already None)."""
    release_model(self, ["model"], device=self.device or "cpu", logger=self.logger)

In [ ]:
#| export
@patch
def _load_model(self:WhisperLocalCapability) -> None:
    """Load the Whisper model (lazy loading)."""
    if self.model is None:
        try:
            self.logger.info(f"Loading Whisper model: {self.config.model}")
            # Heartbeat wraps the WHOLE load: load_model downloads weights from the
            # Azure CDN via urllib on a cold cache (silent to the substrate's stall
            # detector), so the heartbeat keeps the (progress, message) tuple advancing.
            with self.heartbeat("loading Whisper model"):
                self.model = load_model(
                    self.config.model, 
                    device=self.device,
                    download_root=self.model_dir
                )

                # Optionally compile the model (PyTorch 2.0+)
                if self.config.compile_model and hasattr(torch, 'compile'):
                    self.model = torch.compile(self.model)
                    self.logger.info("Model compiled with torch.compile")

            self.logger.info("Local Whisper model loaded successfully")
        except torch.cuda.OutOfMemoryError as e:
            # SG-47 Track B: CUDA OOM during model load -> typed CapabilityResourceError
            # (cjm-substrate-torch-utils); substrate's CR-7 reactive-retry reloads + retries.
            raise cuda_oom_to_capability_resource_error(
                e, label=f"loading Whisper model {self.config.model!r}",
            ) from e
        except Exception as e:
            # SG-47: non-resource load failures (missing model, corrupt download,
            # device misconfiguration) — fatal; author/config attention needed.
            raise CapabilityFatalError(f"Failed to load Whisper model: {e}") from e

In [ ]:
#| export
@patch
def _prepare_audio(
    self:WhisperLocalCapability,
    audio: Union[str, Path] # Path to a decodable audio file
) -> str: # The audio file path
    """Validate the audio input and return it as a path string.

    The caller (orchestration / proxy) guarantees a model-ready audio file;
    in-memory preparation is no longer a capability responsibility."""
    if isinstance(audio, (str, Path)):
        return str(audio)
    raise CapabilityInputError(  # SG-47: typed input-validation (multi-inherits ValueError)
        f"Unsupported audio input type: {type(audio)}; expected a file path (str or Path)",
        fields_invalid=["audio"],
    )

In [ ]:
#| export
@patch
def is_available(self:WhisperLocalCapability) -> bool: # True if Whisper and its dependencies are available
    """Check if Whisper is available."""
    return WHISPER_AVAILABLE

In [ ]:
#| export
@patch
def prefetch(self:WhisperLocalCapability) -> None:
    """CR-4 (SG-19): eagerly load the model so the first execute() doesn't pay
    the download/load cost. Idempotent via _load_model's None-guard."""
    self._load_model()

In [ ]:
#| export
@patch
def on_disable(self:WhisperLocalCapability) -> None:
    """CR-2: release the GPU model when the operator disables the capability (the
    worker stays alive); lazy reload on the next execute after re-enable."""
    self._release_model()

In [ ]:
#| export
@patch
def cleanup(self:WhisperLocalCapability) -> None:
    """Release resources on unload."""
    self._release_model()
    self.logger.info("Cleanup completed")

## Testing the Capability

In [ ]:
# Basic functionality (stage-8: pure-compute ToolCapability).
from cjm_substrate.core.capability import ToolCapability

capability = WhisperLocalCapability()
assert isinstance(capability, ToolCapability)
print(f"Whisper available: {capability.is_available()}")
print(f"Capability version: {capability.version}")
print(f"Config class: {capability.config_class.__name__}")

# Native-surface model: pure-compute `transcribe` replaces the fused `execute`;
# cache/persist + identity moved out. `name` is derived from the installed dist
# at runtime (worker / in-env introspection), so it is NOT exercised in-notebook
# where __package__ is unset.
assert hasattr(capability, "transcribe") and not hasattr(capability, "execute")
assert not hasattr(capability, "supported_formats")
print("WhisperLocalCapability is a pure-compute ToolCapability (transcribe; no execute/cache/storage)")

Whisper available: True
Capability name: whisper_local
Capability version: 1.0.0
Supported formats: ['wav', 'mp3', 'flac', 'm4a', 'ogg', 'webm', 'mp4', 'avi', 'mov']
Config class: WhisperCapabilityConfig


In [ ]:
# Test configuration dataclass
from dataclasses import fields

print("Available models:")
model_field = next(f for f in fields(WhisperCapabilityConfig) if f.name == "model")
for model in model_field.metadata.get(SCHEMA_ENUM, []):
    print(f"  - {model}")

Available models:
  - tiny
  - tiny.en
  - base
  - base.en
  - small
  - small.en
  - medium
  - medium.en
  - large
  - large-v1
  - large-v2
  - large-v3


In [ ]:
# Test configuration validation
test_configs = [
    ({"model": "tiny"}, "Valid config"),
    ({"model": "invalid"}, "Invalid model"),
    ({"model": "tiny", "temperature": 1.5}, "Temperature out of range"),
]

for config, description in test_configs:
    try:
        test_cfg = dict_to_config(WhisperCapabilityConfig, config, validate=True)
        print(f"{description}: Valid=True")
    except ValueError as e:
        print(f"{description}: Valid=False")
        print(f"  Error: {str(e)[:100]}")

Valid config: Valid=True
Invalid model: Valid=False
  Error: model: 'invalid' is not one of ['tiny', 'tiny.en', 'base', 'base.en', 'small', 'small.en', 'medium',
Temperature out of range: Valid=False
  Error: temperature: 1.5 is greater than maximum 1.0


In [ ]:
# Test initialization and get_current_config (returns dict now)
capability.initialize({"model": "tiny", "device": "cpu"})
current_config = capability.get_current_config()
print(f"Current config (dict): {current_config}")
print(f"Current model: {current_config['model']}")

Current config (dict): {'model': 'tiny', 'device': 'cpu', 'language': None, 'task': 'transcribe', 'temperature': 0.0, 'temperature_increment_on_fallback': 0.2, 'beam_size': 5, 'best_of': 5, 'patience': 1.0, 'length_penalty': None, 'suppress_tokens': '-1', 'initial_prompt': None, 'condition_on_previous_text': False, 'fp16': True, 'compression_ratio_threshold': 2.4, 'logprob_threshold': -1.0, 'no_speech_threshold': 0.6, 'word_timestamps': False, 'prepend_punctuations': '"\'“¿([{-', 'append_punctuations': '"\'.。,，!！?？:：”)]}、', 'threads': 0, 'model_dir': None, 'compile_model': False}
Current model: tiny


In [ ]:
#| eval: false
# Test get_config_schema for UI generation
import json

schema = capability.get_config_schema()
print("JSON Schema for WhisperCapabilityConfig:")
print(f"  Name: {schema['name']}")
print(f"  Properties count: {len(schema['properties'])}")
print(f"  Model field enum: {schema['properties']['model'].get('enum', [])[:3]}...")
print(f"\nFull schema (truncated):")
print(json.dumps({k: v for k, v in list(schema['properties'].items())[:3]}, indent=2))

JSON Schema for WhisperCapabilityConfig:
  Name: WhisperCapabilityConfig
  Properties count: 23
  Model field enum: ['tiny', 'tiny.en', 'base']...

Full schema (truncated):
{
  "model": {
    "type": "string",
    "title": "Model",
    "description": "Whisper model size. Larger models are more accurate but slower.",
    "enum": [
      "tiny",
      "tiny.en",
      "base",
      "base.en",
      "small",
      "small.en",
      "medium",
      "medium.en",
      "large",
      "large-v1",
      "large-v2",
      "large-v3"
    ],
    "default": "base"
  },
  "device": {
    "type": "string",
    "title": "Device",
    "description": "Device for inference (auto will use CUDA if available)",
    "enum": [
      "auto",
      "cpu",
      "cuda"
    ],
    "default": "auto"
  },
  "language": {
    "type": [
      "string",
      "null"
    ],
    "title": "Language",
    "description": "Language code (e.g., 'en', 'es', 'fr') or None for auto-detection",
    "default": null
  }
}


In [ ]:
#| eval: false
# Test idempotent initialize - model unload on config change
import logging

# Enable logging to see model unload messages
logging.basicConfig(level=logging.INFO)

# Initialize with one model
capability.initialize({"model": "tiny", "device": "cpu"})
print(f"Initial config: model={capability.config.model}")

# Re-initialize with same model (no unload should happen)
print("\nRe-initializing with same model...")
capability.initialize({"model": "tiny", "device": "cpu"})

# Re-initialize with different model (unload should trigger)
print("\nRe-initializing with different model...")
capability.initialize({"model": "base", "device": "cpu"})
print(f"New config: model={capability.config.model}")

INFO:__main__.WhisperLocalCapability:Initialized Whisper capability with model 'tiny' on device 'cpu'
INFO:__main__.WhisperLocalCapability:Initialized Whisper capability with model 'tiny' on device 'cpu'
INFO:__main__.WhisperLocalCapability:Config change: Model tiny -> base
INFO:__main__.WhisperLocalCapability:Initialized Whisper capability with model 'base' on device 'cpu'


Initial config: model=tiny

Re-initializing with same model...

Re-initializing with different model...
New config: model=base


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()